# Preparando Dados

## Carregando e Lendo Dataset

In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files: /kaggle/input/fashionmnist


In [ ]:
print(os.listdir(path))

['t10k-labels-idx1-ubyte', 't10k-images-idx3-ubyte', 'fashion-mnist_test.csv', 'fashion-mnist_train.csv', 'train-labels-idx1-ubyte', 'train-images-idx3-ubyte']


## Separando Treino e teste

In [ ]:
import pandas as pd
train_df = pd.read_csv(os.path.join(path, 'fashion-mnist_train.csv'))
test_df = pd.read_csv(os.path.join(path, 'fashion-mnist_test.csv'))

print("treino: ", train_df.shape)
print("teste: ", test_df.shape)

treino:  (60000, 785)
teste:  (10000, 785)


In [ ]:
X_train = train_df.drop('label', axis=1)
y_train = train_df['label']

X_test = test_df.drop('label', axis=1)
y_test = test_df['label']

In [ ]:
print(X_train.shape)  # (60000, 784)
print(y_train.shape)  # (60000,)

(60000, 784)
(60000,)


In [ ]:
#Normalização
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_mn = scaler.fit_transform(X_train)
X_test_mn = scaler.transform(X_test)

#Base Line

Decision Tree Classifier utilizando todas as Features

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
clf = DecisionTreeClassifier(random_state=42)

clf.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [ ]:
all_feature_names = clf.feature_names_in_

In [ ]:
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.7963


Acurácia no conjunto de teste: 0.7963
'
Porcentagem de features selecionadas (em relação aos dados originais): 100%

Tempo de treinamento: 1min

Tempo necessário para encontrar o subconjunto de features: N/A (usaram-se todas)


# Genetic Algorithm

In [ ]:
import time
import random
import sys
import pickle
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score

## Ferramentas

In [ ]:
def append_list_items(listA: list, listB: list):
  """
  listA irá receber todos os itens de listB
  """
  for i in range(len(listB) - 1):
    listA.append(listB[i])

In [ ]:
def load_tree(gen: int, name: int):
  file_name = f"tree{gen}{name}.dat"
  pk = open(file_name, "rb")

  tree = pickle.load(pk)
  pk.close()

  return tree

In [ ]:
def create_tree_file(gen: int, name: int):
  file_name = f"tree{gen}{name}.dat"
  tree = open(file_name, "w+b")
  tree.close()

  return file_name

In [ ]:
def create_features_file(gen: int, name: int):
  file_name = f"features_{gen}{name}.dat"
  feats = open(file_name, "w+b")
  feats.close()

  return file_name

## Fitness Function

In [ ]:
def fitness(clf: DecisionTreeClassifier, acc: float):
  f = clf.n_features_in_
  return (acc * 100) - (f / 10)

## Crossover

In [ ]:
def mutate(genes: np.ndarray, what_gen: int):
  random.seed()

  print("Quantidade features: {}".format(genes.size))
  # Chance de mutação: 1/tamanho do gene
  if ((random.randint(0, (genes.size -1) )) == 1):
    rng = np.random.default_rng()

    if (genes.size  >= 100):
      new_genes_qt = int(genes.size  /100)
      new_genes = rng.choice(all_feature_names, size=new_genes_qt, replace=False)

      for i in range((genes.size -1), (genes.size - new_genes_qt - 1), -1):
        genes = np.delete(genes.size , i)
      genes.append(new_genes)

      # para genes curtos
    else:
      new_genes = rng.choice(all_feature_names, size=1, replace=False)
      genes[genes.size-1] = new_genes

  # Randomizar TUDO na primeira gen
  elif(what_gen == 0):
    new_genes_qt = int(genes.size)
    new_genes = rng.choice(all_feature_names, size=new_genes_qt, replace=False)

    genes = new_genes

  return genes

In [ ]:
def crossover(parentA: DecisionTreeClassifier, parentB: DecisionTreeClassifier, what_gen: int, current_pop: int):
  """
  Cria dois filhos utlizando crossover de um ponto.
  Os filhos são treinados com as features designadas.

  return:
  filho1: DecisionTreeClassifier,
  filho2: DecisionTreeClassifier
  """
  training_time = 0
  # Carregando pais
  A_tree = load_tree(what_gen, parentA)
  B_tree = load_tree(what_gen, parentB)

  # Separação dos genes
  A_gene = np.split(A_tree.feature_names_in_, 2)
  B_gene = np.split(B_tree.feature_names_in_, 2)

  # Filho 1 -----------------------
  offspring1 = DecisionTreeClassifier(random_state=42)
  genes_offspring1 = np.concatenate((A_gene[0], B_gene[1]))

  # Mutação
  mutate(genes_offspring1, what_gen)
  genes_offspring1 = np.unique(genes_offspring1)

  X_train_1 = X_train[genes_offspring1]

  timer_s = time.time()
  offspring1.fit(X_train_1, y_train)
  timer_e = time.time()

  tree = create_tree_file(what_gen, (current_pop + 1))
  with open(tree, "wb") as f:
        pickle.dump(offspring1, f)
  f.close()

  training_time += timer_e - timer_s

  # Filho 2 -----------------------
  offspring2 = DecisionTreeClassifier(random_state=42)
  genes_offspring2 = np.concatenate((A_gene[1], B_gene[0]))
  # Mutação
  mutate(genes_offspring2, what_gen)
  genes_offspring2 = np.unique(genes_offspring2)

  X_train_2 = X_train[genes_offspring2]

  timer_s = time.time()
  offspring2.fit(X_train_2, y_train)
  timer_e = time.time()


  tree = create_tree_file(what_gen, (current_pop + 2))
  with open(tree, "wb") as f:
        pickle.dump(offspring2, f)
  f.close()

  training_time += timer_e - timer_s
  return training_time

In [ ]:
# Seleção de pais por método de roleta
def parent_selection(fitness_scores: list, population: int):

  random.seed()
  S = sum(fitness_scores)
  score_ammount = len(fitness_scores)
  fixed_arrow = random.uniform(0.0, S)

  # Roda a roleta
  i = 0
  spinning_sum = 0
  while((spinning_sum < fixed_arrow) and (i < (score_ammount-1))):
    spinning_sum += fitness_scores[i]
    i += 1

  parentA = i
  fitness_scores.pop(i)
  score_ammount -= 1

  S = sum(fitness_scores)
  fixed_arrow = random.uniform(0.0, S)

  # Roda a roleta denovo
  i = 0
  spinning_sum = 0
  while((spinning_sum < fixed_arrow) and (i < (score_ammount-1))):
    spinning_sum += fitness_scores[i]
    i += 1

  parentB = i

  return parentA, parentB

## Geração

In [ ]:
def generation(population: int, what_gen: int):
  """
  Realiza os testes dos candidatos.
  gera candidatos na memória

  population: Tamanho da população, int
  candidates: Lista de candidatos DecisionTreeClassifier

  return:
  fitness_scores: Lista do fitness score dos candidatos.
  accuracy_scores: Lista da acurácia dos candidatos.
  """
  fitness_scores = []
  accuracy_scores = []
  training_time = 0

  if (what_gen > 0):
    for i in range(population):
      candidate = load_tree(i)

      pred = candidate.predict(X_test)
      acc = accuracy_score(y_test, pred)
      f = fitness(candidate, acc)

      fitness_scores.append(f)
      accuracy_scores.append(acc)

  # Geração 1
  else:
    for i in range(population):
      tree = create_tree_file(what_gen, i)

      timer_s = time.time()
      clf = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
      timer_e = time.time()

      with open(tree, "wb") as f:
        pickle.dump(clf, f)
      f.close()

      pred = clf.predict(X_test)

      acc = accuracy_score(y_test, pred)
      f = fitness(clf, acc)

      fitness_scores.append(f)
      accuracy_scores.append(acc)
      training_time += (timer_e - timer_s)

  return fitness_scores, accuracy_scores, training_time

## Algoritmo

In [ ]:
def GeneticAlgorithm_FS(generations: int, population: int):
  """
  docstring
  """
  s = time.time()

  total_training_time = 0
  candidates = []
  all_fitness = []
  all_accuracy = []
  best_gen_fitness_i = []

  for i in range(generations):
      gen_number = i

      timer_s = time.time()
      fitness_scores, accuracy_scores, training_time = generation(population, gen_number)
      timer_e = time.time()


      mean_gf = np.mean(fitness_scores)
      M_gf = max(fitness_scores)
      mean_gacc = np.mean(accuracy_scores)
      M_gacc = max(accuracy_scores)

      gen_time = timer_e - timer_s

      total_training_time += training_time

      best_gen_fitness_i.append(fitness_scores.index(M_gf))
      append_list_items(all_fitness, fitness_scores)
      append_list_items(all_accuracy, accuracy_scores)
      print("gen: mean fitness: max fitness:  mean accuracy: max accuracy:  time elapsed: ")
      print("{gen}    {fit:.5f}       {fitM:.5f}       {acc:.5f}          {accM:.5f}     {time}".format(gen=i, fit=mean_gf, fitM=M_gf, acc=mean_gacc, accM=M_gacc, time=gen_time))

      # Geração de novos candidatos
      for i in range(0, population, 2):
        parentA, parentB = parent_selection(fitness_scores, population)
        c_train_time = crossover(parentA, parentB, gen_number, population)

        total_training_time += c_train_time

      # fim iteração

  e = time.time()
  f = max(all_fitness)
  a = max(all_accuracy)
  total_time = e - s
  print("best fitness:      best accuracy:     Training time:      Feature search time:")
  print("{fit:.5f}          {acc:.5f}          {t1}                {t2} sec.".format(gen=i, fit=f, acc=a, t1=total_training_time, t2=total_time))



## Execução

In [ ]:
GeneticAlgorithm_FS(2, 2)

gen: mean fitness: max fitness:  mean accuracy: max accuracy:  time elapsed: 
0    1.23000       1.23000       0.79630          0.79630     545.8282577991486
Quantidade features: 784
Quantidade features: 784
Quantidade features: 784
Quantidade features: 784
Quantidade features: 784
Quantidade features: 784
Quantidade features: 784
Quantidade features: 784
Quantidade features: 784
Quantidade features: 784


TypeError: load_tree() missing 1 required positional argument: 'name'